# Example 13 — Direction-Finding Dataset with MUSIC and ESPRIT

**Level:** Intermediate

After working through this notebook you will know how to:

- Configure a Uniform Linear Array (ULA) using `spectra.arrays.ula()`
- Build a `DirectionFindingDataset` with multiple concurrent sources
- Convert multi-antenna IQ snapshots to a complex snapshot matrix
- Estimate source azimuths with the **MUSIC** pseudospectrum algorithm
- Estimate source azimuths with the **ESPRIT** subspace algorithm
- Compare estimated angles to dataset ground truth and compute RMSE

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from spectra.arrays import ula
from spectra.datasets import DirectionFindingDataset
from spectra.waveforms import BPSK, QPSK, QAM16
from spectra.algorithms import music, esprit, find_peaks_doa
from plot_helpers import savefig

# ── Configuration ─────────────────────────────────────────────────────────────
FREQ_HZ      = 2.4e9
N_ELEMENTS   = 8
SPACING      = 0.5
N_SNAPSHOTS  = 256
SAMPLE_RATE  = 1e6
N_SOURCES    = 2
SNR_RANGE    = (15.0, 25.0)
N_SAMPLES    = 200
SEED         = 42

SCAN_DEG = np.linspace(5, 175, 512)
SCAN_RAD = np.deg2rad(SCAN_DEG)

## 1. Array Configuration

`ula()` creates a Uniform Linear Array (ULA) along the x-axis with `N` isotropic elements
separated by `d` wavelengths. The default half-wavelength spacing (`d = 0.5λ`) avoids
spatial aliasing for sources across the full visible range `[0°, 180°]`.

The `AntennaArray` object stores element positions in wavelengths and provides
`steering_vector(az, el)` for computing the array manifold vector at any direction.

In [ ]:
arr = ula(num_elements=N_ELEMENTS, spacing=SPACING, frequency=FREQ_HZ)
print(f"ULA: {arr.num_elements} elements, {SPACING}λ spacing")
print(f"Element positions (wavelengths):")
print(f"  x = {arr.positions[:, 0]}")

## 2. Dataset Creation

`DirectionFindingDataset` generates multi-antenna IQ snapshots on-the-fly. Each item
contains `num_signals` sources drawn from `signal_pool`, placed at randomly sampled azimuths
subject to the `min_angular_separation` constraint. The dataset uses deterministic
`(seed, idx)` seeding so it is safe to use with multiple DataLoader workers.

**Output shape:** `Tensor[N_elements, 2, num_snapshots]` (float32; channel 0 = I, channel 1 = Q)

**Target:** `DirectionFindingTarget` dataclass with `azimuths`, `elevations`, `snrs`, `labels`.

In [ ]:
signal_pool = [
    BPSK(samples_per_symbol=4),
    QPSK(samples_per_symbol=4),
    QAM16(samples_per_symbol=4),
]

ds = DirectionFindingDataset(
    array=arr,
    signal_pool=signal_pool,
    num_signals=N_SOURCES,
    num_snapshots=N_SNAPSHOTS,
    sample_rate=SAMPLE_RATE,
    snr_range=SNR_RANGE,
    azimuth_range=(np.deg2rad(10), np.deg2rad(170)),
    elevation_range=(0.0, 0.0),
    min_angular_separation=np.deg2rad(15),
    num_samples=N_SAMPLES,
    seed=SEED,
)
print(f"Dataset size: {len(ds)} samples, {N_SOURCES} sources each")
data, target = ds[0]
print(f"Data shape: {data.shape}")
print("\nSample 0 ground truth:")
for k in range(target.num_sources):
    print(f"  Source {k}: az = {np.rad2deg(target.azimuths[k]):.1f}°, "
          f"SNR = {target.snrs[k]:.1f} dB, label = {target.labels[k]}")

## 3. Array Geometry Visualisation

For a ULA along the x-axis, the `i`-th element is at position `(i · d, 0)` in wavelengths.
The broadside direction (maximum array gain) is at 90° from the x-axis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2))
ax.scatter(arr.positions[:, 0], arr.positions[:, 1], s=120, zorder=3)
for i, (x, y) in enumerate(arr.positions):
    ax.annotate(f"{i}", (x, y), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("x (wavelengths)")
ax.set_title(f"ULA Geometry — {N_ELEMENTS} elements, d = {SPACING}λ")
ax.set_yticks([])
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig("13_array_geometry.png")
plt.show()

## 4. Inspect One Sample

Each element receives a superposition of `N_SOURCES` delayed-and-phase-shifted copies of the
source signals plus additive white Gaussian noise. The phase progression across elements
encodes the source directions.

In [ ]:
fig, axes = plt.subplots(N_ELEMENTS, 1, figsize=(10, 12), sharex=True)
t = np.arange(N_SNAPSHOTS) / SAMPLE_RATE * 1e6  # µs
for i, ax in enumerate(axes):
    iq = data[i, 0, :].numpy()      # I channel
    ax.plot(t, iq, linewidth=0.6)
    ax.set_ylabel(f"Ant {i}", fontsize=7)
    ax.set_yticks([])
axes[-1].set_xlabel("Time (µs)")
fig.suptitle("Received I-channel per antenna element (sample 0)")
plt.tight_layout()
savefig("13_snapshot_iq.png")
plt.show()

## 5. MUSIC Pseudospectrum

**MUSIC** (MUltiple SIgnal Classification, Schmidt 1986) exploits the orthogonality between
the signal subspace and noise subspace of the sample covariance matrix.

Given snapshot matrix **X** ∈ ℂ^(N×T):

1. Form the sample covariance: **R** = **X** **X**^H / T
2. Eigendecompose: **R** = **U** **Λ** **U**^H (eigenvalues ascending)
3. Extract noise subspace: **E**_n = **U**[:, :N−K] (smallest N−K eigenvectors)
4. Evaluate pseudospectrum: P(θ) = 1 / ‖**E**_n^H **a**(θ)‖²

Source azimuths appear as sharp peaks in P(θ) because **a**(θ_true) is orthogonal to **E**_n.

> Pisarenko, V. F. (1973). _The retrieval of harmonics from a covariance function._ Geophysical Journal International, 33(3), 347–366.  
> Schmidt, R. O. (1986). _Multiple emitter location and signal parameter estimation._ IEEE Trans. Antennas and Propagation, 34(3), 276–280.

In [ ]:
# Convert [N_elem, 2, T] → complex [N_elem, T]
X = data[:, 0, :].numpy() + 1j * data[:, 1, :].numpy()   # (N, T)

spectrum = music(X, num_sources=N_SOURCES, array=arr, scan_angles=SCAN_RAD, elevation=0.0)
peaks_music = find_peaks_doa(spectrum, SCAN_RAD, num_peaks=N_SOURCES)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(SCAN_DEG, 10 * np.log10(spectrum / spectrum.max()), color="steelblue", linewidth=1.2)
for k, az_true in enumerate(target.azimuths):
    ax.axvline(np.rad2deg(az_true), color="crimson", linestyle="--", linewidth=1.5,
               label=f"True az {np.rad2deg(az_true):.1f}°")
for az_est in peaks_music:
    ax.axvline(np.rad2deg(az_est), color="orange", linestyle=":", linewidth=1.5)
ax.set_xlabel("Azimuth (degrees from x-axis)")
ax.set_ylabel("Pseudospectrum (dB, normalised)")
ax.set_title(f"MUSIC — {N_ELEMENTS}-element ULA, {N_SOURCES} sources, SNR ∈ {SNR_RANGE} dB")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig("13_music_spectrum.png")
plt.show()

print(f"MUSIC peaks:   {np.rad2deg(peaks_music)}")
print(f"True azimuths: {np.rad2deg(np.sort(target.azimuths))}")

## 6. ESPRIT

**ESPRIT** (Estimation of Signal Parameters via Rotational Invariance Techniques, Roy & Kailath 1989)
exploits the shift-invariance of a ULA. Two overlapping subarrays share a common
signal subspace rotated by a diagonal phase matrix **Φ**:

**E**_s2 ≈ **E**_s1 **Φ**,    **Φ** = diag(e^{j·2π·d·cos(θ_k)})

Solving for **Φ** via least squares and taking eigenvalues gives the inter-element
phase shifts directly, so **no spatial scanning is needed** — ESPRIT is typically faster
than MUSIC.

> **Note:** ESPRIT requires a shift-invariant (ULA) array. It cannot be applied to
> non-uniform or circular arrays without modification.

> Roy, R., & Kailath, T. (1989). _ESPRIT — Estimation of signal parameters via rotational
> invariance techniques._ IEEE Trans. Acoustics, Speech, and Signal Processing, 37(7), 984–995.

In [ ]:
az_esprit = esprit(X, num_sources=N_SOURCES, spacing=SPACING)
print(f"ESPRIT estimates: {np.rad2deg(az_esprit)}")
print(f"True azimuths:    {np.rad2deg(np.sort(target.azimuths))}")

## 7. Batch RMSE Comparison

To get a statistically meaningful comparison we iterate over 100 dataset samples and
compute per-source angular error for both algorithms, then report Root Mean Square Error (RMSE)
in degrees.

In [ ]:
def collate_fn(batch):
    return torch.stack([x for x, _ in batch]), [t for _, t in batch]

loader = DataLoader(ds, batch_size=1, collate_fn=collate_fn)

music_errors, esprit_errors = [], []

for i, (batch_data, batch_targets) in enumerate(loader):
    if i >= 100:
        break
    tgt = batch_targets[0]
    Xb = batch_data[0, :, 0, :].numpy() + 1j * batch_data[0, :, 1, :].numpy()
    true_az = np.sort(tgt.azimuths)

    # MUSIC
    sp_val = music(Xb, num_sources=N_SOURCES, array=arr, scan_angles=SCAN_RAD)
    est_music = find_peaks_doa(sp_val, SCAN_RAD, num_peaks=N_SOURCES)
    for j, az_t in enumerate(true_az):
        music_errors.append(abs(est_music[j % len(est_music)] - az_t))

    # ESPRIT
    est_esprit = esprit(Xb, num_sources=N_SOURCES, spacing=SPACING)
    for j, az_t in enumerate(true_az):
        esprit_errors.append(abs(est_esprit[j % len(est_esprit)] - az_t))

rmse_music  = np.rad2deg(np.sqrt(np.mean(np.array(music_errors)  ** 2)))
rmse_esprit = np.rad2deg(np.sqrt(np.mean(np.array(esprit_errors) ** 2)))
print(f"RMSE over 100 samples:")
print(f"  MUSIC:  {rmse_music:.2f}°")
print(f"  ESPRIT: {rmse_esprit:.2f}°")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["MUSIC", "ESPRIT"], [rmse_music, rmse_esprit], color=["steelblue", "seagreen"])
ax.set_ylabel("RMSE (degrees)")
ax.set_title(f"DoA Estimation RMSE — 100 samples, SNR ∈ {SNR_RANGE} dB")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig("13_rmse_comparison.png")
plt.show()